In [1]:
import sys
import json
from pathlib import Path

sys.path.insert(0, "./../")

from graph_retriever import GraphRetriever
from fact_checker import FactChecker
from qa_generator import QAGenerator


def strip_code_fence(text: str) -> str:
    if text.startswith("```json"):
        return text.split("```json", 1)[1].rsplit("```", 1)[0].strip()
    if text.startswith("```"):
        return text.split("```", 1)[1].rsplit("```", 1)[0].strip()
    return text.strip()


def parse_json_result(text: str, fallback):
    try:
        return json.loads(strip_code_fence(text))
    except Exception:
        return fallback

In [3]:
STEP3_OUT = "../../TMP/wcep_kg_4/step3_output.jsonl"
STEP4_OUT = "../../TMP/wcep_kg_4/step4_output.jsonl"
MODEL_NAME = "qwen3.5-flash"

In [4]:
with open(STEP3_OUT, "r", encoding="utf-8") as f:
    samples = [json.loads(line) for line in f if line.strip()]

print(f"Loaded {len(samples)} samples from {STEP3_OUT}")

Loaded 1 samples from ../../TMP/wcep_kg_4/step3_output.jsonl


## Load Step3 Output

## Graph Retriever

In [5]:
for sample in samples:
    all_triplets = sample.get("all_triplets", [])
    kg_index = sample.get("sample_entity_index", {})
    aligned_map = sample.get("entity_alignment", {})
    aligned_entities = list(aligned_map.values())

    retriever = GraphRetriever(all_triplets, kg_index)
    retrieved_triplets = retriever.retrieve_by_entities(aligned_entities)
    sample["retrieved_triplets"] = [list(t) for t in retrieved_triplets]

print(f"Graph retriever finished for {len(samples)} samples")

Graph retriever finished for 1 samples


## Fact Checker

In [6]:
checker = FactChecker(model_name=MODEL_NAME)


def run_fact_checker(checker, claim: str, retrieved_triplets: list) -> dict:
    triplets_str = "\n".join([f"({t[0]}, {t[1]}, {t[2]})" for t in retrieved_triplets])

    prompt = f"""You are a helpful assistant for fact-checking a claim against a list of retrieved knowledge graph triplets. Your task is to determine whether the claim is supported, refuted, or if there is not enough information (NEI) based on the provided triplets.
Claim: \"{claim}\"
Retrieved Knowledge Graph Triplets:
{triplets_str if triplets_str else "No triplets retrieved."}
Please strictly analyze the claim in relation to the retrieved triplets and provide a verdict of \"SUPPORT\", \"PARTIALLY_SUPPORTED\", \"PARTIALLY_CONTRADICTED\", \"CONTRADICTED\", or \"NEI\".
You must return the expected JSON object without any additional explanations or markdown formatting:
{{
    "verdict": "SUPPORT/PARTIALLY_SUPPORTED/PARTIALLY_CONTRADICTED/CONTRADICTED/NEI",
    "confidence": 0.85,
    "explanation": "A brief explanation of the reasoning behind the verdict.",
    "key_evidence": "The most relevant triplet(s) that support the verdict, if any."
}}
"""
    raw_text = checker.generator._call_qwen(
        role="You are a helpful assistant for fact-checking.",
        prompt=prompt,
    )
    default_result = {
        "verdict": "NEI",
        "confidence": 0.0,
        "explanation": "Fact-checking failed.",
        "key_evidence": "",
    }
    return parse_json_result(raw_text, default_result)


for sample in samples:
    claim_text = sample.get("claim_text", "")
    sample["fact_check_result"] = run_fact_checker(
        checker,
        claim_text,
        sample.get("retrieved_triplets", []),
    )

print(f"Fact checker finished for {len(samples)} samples")

Fact checker finished for 1 samples


In [7]:
def run_fact_checker(checker, claim: str, retrieved_triplets: list) -> dict:
    triplets_str = "\n".join([f"({t[0]}, {t[1]}, {t[2]})" for t in retrieved_triplets])

    prompt = f"""You are a helpful assistant for fact-checking a claim against a list of retrieved knowledge graph triplets. Your task is to determine whether the claim is supported, refuted, or if there is not enough information (NEI) based on the provided triplets.
Claim: \"{claim}\"
Retrieved Knowledge Graph Triplets:
{triplets_str if triplets_str else "No triplets retrieved."}
Please strictly analyze the claim in relation to the retrieved triplets and provide a verdict of \"SUPPORT\", \"PARTIALLY_SUPPORTED\", \"PARTIALLY_CONTRADICTED\", \"CONTRADICTED\", or \"NEI\".
You must return the expected JSON object without any additional explanations or markdown formatting:
{{
    "verdict": "SUPPORT/PARTIALLY_SUPPORTED/PARTIALLY_CONTRADICTED/CONTRADICTED/NEI",
    "confidence": 0.85,
    "explanation": "A brief explanation of the reasoning behind the verdict.",
    "key_evidence": "The most relevant triplet(s) that support the verdict, if any."
}}
"""
    raw_text = checker.generator._call_qwen(
        role="You are a helpful assistant for fact-checking.",
        prompt=prompt,
    )
    default_result = {
        "verdict": "NEI",
        "confidence": 0.0,
        "explanation": "Fact-checking failed.",
        "key_evidence": "",
    }
    return parse_json_result(raw_text, default_result)


for sample in samples:
    claim_text = sample.get("claim_text", "")
    sample["fact_check_result"] = run_fact_checker(
        checker,
        claim_text,
        sample.get("retrieved_triplets", []),
    )

print(f"Fact checker finished for {len(samples)} samples")

Fact checker finished for 1 samples


## QA Generator

In [9]:
qa_gen = QAGenerator(model_name=MODEL_NAME)


def run_qa_generator(qa_gen, claim: str, check_result: dict) -> list:
    prompt = f"""You need to generate question-answer pairs based on a claim, its fact-checking verdict, and key evidence. The fact-checking result is provided in the following JSON format:{json.dumps(check_result, ensure_ascii=False)}
Based on the claim, the fact-checking verdict, and the key evidence, please generate question-answer pairs in the following four dimensions:
1. Comprehension Questions: Questions that test understanding of the claim and its context.
2. Evidence-Based Questions: Questions that require using the key evidence to answer.
3. Counterfactual Questions: Questions that explore hypothetical scenarios related to the claim.
4. Critical Thinking Questions: Questions that encourage analysis and evaluation of the claim and evidence.
Please return a JSON array of question-answer pairs, where each pair includes the question, the answer, and the dimension it belongs to. Do not include any additional explanations or formatting.
Expected Output Format:
[
    {{
        "question": "What is the main claim being evaluated?",
        "answer": "The main claim is that ...",
        "dimension": "Comprehension"
    }},
    {{
        "question": "What evidence supports the claim?",
        "answer": "The key evidence supporting the claim is ...",
        "dimension": "Evidence-Based"
    }},
    {{
        "question": "What if the key evidence was different?",
        "answer": "If the key evidence was different, then ...",
        "dimension": "Counterfactual"
    }},
    {{
        "question": "How would you evaluate the strength of the claim based on the evidence?",
        "answer": "The strength of the claim can be evaluated as ...",
        "dimension": "Critical Thinking"
    }}
]"""
    raw_text = qa_gen.generator._call_qwen(
        role="You are a helpful assistant for generating question-answer pairs.",
        prompt=prompt,
    )
    return parse_json_result(raw_text, [])


for sample in samples:
    check_result = sample.get("fact_check_result", {})
    sample["generated_qa_pairs"] = run_qa_generator(
        qa_gen,
        sample.get("claim_text", ""),
        check_result,
    )

print(f"QA generation finished for {len(samples)} samples")

QA generation finished for 1 samples


In [11]:
with open(STEP4_OUT, "w", encoding="utf-8") as out_f:
    for sample in samples:
        out_f.write(json.dumps(sample, ensure_ascii=False) + "\n")

print(f"Saved {len(samples)} samples to {STEP4_OUT}")

Saved 1 samples to ../../TMP/wcep_kg_4/step4_output.jsonl
